# Polymarket Trading Bot — Quickstart

Run each cell in order to verify your setup and explore your account state.

**Prerequisites:** Fill in your credentials in `.env` at the project root (see `.env.example`).

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env and add project root to path
load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('..').resolve()))

print('Environment loaded.')

In [ ]:
# Check required environment variables
private_key = os.environ.get('POLY_PRIVATE_KEY')
safe_address = os.environ.get('POLY_SAFE_ADDRESS')

assert private_key, 'POLY_PRIVATE_KEY is not set in .env'
assert safe_address, 'POLY_SAFE_ADDRESS is not set in .env'

print(f'Safe address : {safe_address}')
print(f'Private key  : {private_key[:6]}...{private_key[-4:]}  (truncated)')

In [ ]:
from src.config import Config

config = Config.from_env()
print(f'Gasless mode : {config.use_gasless}')
print(f'Chain ID     : {config.clob.chain_id}')
print(f'CLOB host    : {config.clob.host}')

In [ ]:
from src.bot import TradingBot

bot = TradingBot(config=config, private_key=private_key)

assert bot.is_initialized(), 'Bot failed to initialize'
print(f'Bot initialized!')
print(f'Signer address : {bot.signer.address}')

In [ ]:
# Fetch open orders
orders = await bot.get_open_orders()
print(f'Open orders: {len(orders)}')
for order in orders[:5]:
    token = order.get('tokenId', '?')[:16]
    side  = order.get('side', '?')
    price = order.get('price', '?')
    size  = order.get('size', '?')
    print(f'  {side} {size} @ {price}  (token: {token}...)')

In [ ]:
trades = await bot.get_trades(limit=10)
print(trades[0])

In [ ]:
# Fetch recent trades
trades = await bot.get_trades(limit=10)
print(f'Recent trades: {len(trades)}')
for trade in trades:
    side  = trade.get('side', '?')
    price = trade.get('price', '?')
    size  = trade.get('size', '?')
    value = float(size) * float(price)
    print(f'  {side} {size} shares @ {price}  = ${value:.2f}')

## Next Steps

```python
# Place an order
result = await bot.place_order(token_id, price=0.5, size=1.0, side='BUY')
print(result.order_id if result.success else result.message)

# Cancel an order
await bot.cancel_order(order_id)

# Cancel all orders
await bot.cancel_all_orders()
```

See `examples/basic_trading.py` for more, or `apps/run_flash_crash.py` to run a strategy.